In [1]:
from pathlib import Path
from remotezip import RemoteZip
from urllib.request import urlretrieve
import zipfile
import json

import numpy as np
import pandas as pd

In [2]:
MAMMALPS_URL = "https://zenodo.org/records/15588220/files/mammalps_v1.zip"
DENSE_URL = "https://zenodo.org/records/15588220/files/dense_annotations_v1.zip"

META_DIR = Path("metadata")
B2_META_DIR = META_DIR / "benchmark_2"
DATASET_META_DIR = META_DIR / "dataset"

B2_META_DIR.mkdir(parents=True, exist_ok=True)
DATASET_META_DIR.mkdir(parents=True, exist_ok=True)

B2_TRAIN_PATH = B2_META_DIR / "metadata_dtci_r11_train.csv"
B2_TEST_PATH = B2_META_DIR / "metadata_dtci_r11_test.csv"
RAW_META_PATH = DATASET_META_DIR / "raw_videos_mammalps_v1.csv"
DENSE_ZIP = META_DIR / "dense_annotations_v1.zip"

DENSE_SUMMARY_PATH = META_DIR / "dense_summary.csv"
EVENT_MANIFEST_PATH = META_DIR / "benchmark_2_event_manifest.csv"
VISIBLE_INTERVALS_PATH = META_DIR / "benchmark_2_visible_intervals.csv"
PRIMARY_MANIFEST_PATH = META_DIR / "benchmark_2_audio_primary_manifest.csv"

In [3]:
required_remote_files = {
    "benchmark_2/metadata/dtci_r11/train.csv": B2_TRAIN_PATH,
    "benchmark_2/metadata/dtci_r11/test.csv": B2_TEST_PATH,
    "raw_videos_mammalps_v1.csv": RAW_META_PATH,
}

missing = {
    remote: local
    for remote, local in required_remote_files.items()
    if not local.exists()
}

if missing:
    with RemoteZip(MAMMALPS_URL) as z:
        for remote_path, local_path in missing.items():
            local_path.parent.mkdir(parents=True, exist_ok=True)
            local_path.write_bytes(z.read(remote_path))
            print("Downloaded:", local_path)
else:
    print("Benchmark II and raw-video metadata already available.")

if not DENSE_ZIP.exists():
    urlretrieve(DENSE_URL, DENSE_ZIP)
    print("Downloaded:", DENSE_ZIP)
else:
    print("Dense annotations already available:", DENSE_ZIP)

Benchmark II and raw-video metadata already available.
Dense annotations already available: metadata\dense_annotations_v1.zip


In [4]:
B2_COLUMNS = [
    "event_id",
    "feature_path",
    "activities",
    "species",
    "weather",
    "num_individuals",
]

b2_train = pd.read_csv(
    B2_TRAIN_PATH,
    sep=r"\s+",
    header=None,
    names=B2_COLUMNS,
    dtype=str,
)

b2_test = pd.read_csv(
    B2_TEST_PATH,
    sep=r"\s+",
    header=None,
    names=B2_COLUMNS,
    dtype=str,
)

b2 = pd.concat(
    [
        b2_train.assign(split="train"),
        b2_test.assign(split="test"),
    ],
    ignore_index=True,
)

raw_meta = pd.read_csv(RAW_META_PATH)

print("B2 events:", len(b2))
print("Raw videos:", len(raw_meta))

B2 events: 397
Raw videos: 2459


In [5]:
background_mask = (
    (b2["activities"] == "none")
    & (b2["species"] == "none")
    & (b2["num_individuals"] == "0")
)

b2_manifest = b2.copy()
b2_manifest["binary_label"] = (~background_mask).astype(int)
b2_manifest["label_name"] = b2_manifest["binary_label"].map(
    {0: "background", 1: "animal"}
)

print(b2_manifest["label_name"].value_counts())

positive_event_ids = set(
    b2_manifest.loc[b2_manifest["binary_label"] == 1, "event_id"]
)

label_name
animal        319
background     78
Name: count, dtype: int64


In [6]:
VISIBLE_GAP_TOLERANCE_S = 0.5

dense_rows = []
visible_interval_rows = []

with zipfile.ZipFile(DENSE_ZIP) as z:
    dense_json_files = [
        name for name in z.namelist()
        if name.lower().endswith(".json")
    ]

    for json_path in dense_json_files:
        with z.open(json_path) as f:
            annotation = json.load(f)

        info = annotation["info"]
        frames = annotation["frames"]

        detected_frames = [
            frame["frame_id"]
            for frame in frames
            if len(frame["detections"]) > 0
        ]

        dense_rows.append({
            "file_id": info["file_id"],
            "event_id": info["event_id"],
            "site_id": info["site_id"],
            "cam_id": info["cam_id"],
            "duration_s": info["duration_s"],
            "fps": info["fps"],
            "automatically_discarded": info["automatically_discarded"],
            "num_frames": len(frames),
            "frames_with_detection": len(detected_frames),
            "total_detections": sum(len(frame["detections"]) for frame in frames),
            "has_detection": len(detected_frames) > 0,
        })

        if info["event_id"] not in positive_event_ids or not detected_frames:
            continue

        fps = float(info["fps"])
        max_gap_frames = int(VISIBLE_GAP_TOLERANCE_S * fps)

        start_frame = detected_frames[0]
        previous_frame = detected_frames[0]

        for frame_id in detected_frames[1:]:
            if frame_id - previous_frame > max_gap_frames + 1:
                start_s = start_frame / fps
                end_s = (previous_frame + 1) / fps

                visible_interval_rows.append({
                    "event_id": info["event_id"],
                    "file_id": info["file_id"],
                    "site_id": info["site_id"],
                    "cam_id": info["cam_id"],
                    "start_s": start_s,
                    "end_s": end_s,
                    "visible_duration_s": end_s - start_s,
                    "video_duration_s": float(info["duration_s"]),
                })
                start_frame = frame_id

            previous_frame = frame_id

        start_s = start_frame / fps
        end_s = (previous_frame + 1) / fps
        visible_interval_rows.append({
            "event_id": info["event_id"],
            "file_id": info["file_id"],
            "site_id": info["site_id"],
            "cam_id": info["cam_id"],
            "start_s": start_s,
            "end_s": end_s,
            "visible_duration_s": end_s - start_s,
            "video_duration_s": float(info["duration_s"]),
        })

dense_summary = pd.DataFrame(dense_rows)
visible_intervals = pd.DataFrame(visible_interval_rows)

print("Dense videos:", len(dense_summary))
print("Visible intervals:", len(visible_intervals))
print("Positive events represented:", visible_intervals["event_id"].nunique())

Dense videos: 2459
Visible intervals: 1681
Positive events represented: 319


In [7]:
dense_event_summary = (
    dense_summary
    .groupby("event_id", as_index=False)
    .agg(
        dense_num_videos=("file_id", "nunique"),
        videos_with_detection=("has_detection", "sum"),
        total_detections=("total_detections", "sum"),
        discarded_videos=("automatically_discarded", "sum"),
    )
)

raw_event_summary = (
    raw_meta
    .groupby("event_id", as_index=False)
    .agg(
        num_videos=("file_id", "nunique"),
        total_duration_s=("duration", "sum"),
        site_ids=("site_id", lambda x: sorted(x.dropna().unique().tolist())),
        camera_ids=("cam_id", lambda x: sorted(x.dropna().unique().tolist())),
        file_ids=("file_id", lambda x: x.tolist()),
        file_paths=("file_path", lambda x: x.tolist()),
    )
)

event_manifest = (
    b2_manifest
    .merge(raw_event_summary, on="event_id", how="left")
    .merge(dense_event_summary, on="event_id", how="left")
)

event_manifest["all_videos_discarded"] = (
    event_manifest["discarded_videos"] == event_manifest["num_videos"]
)

In [8]:
background_events = event_manifest["label_name"] == "background"
animal_events = event_manifest["label_name"] == "animal"

assert len(event_manifest) == 397
assert event_manifest["event_id"].nunique() == 397
assert event_manifest["num_videos"].notna().all()
assert event_manifest["total_detections"].notna().all()
assert (event_manifest.loc[background_events, "videos_with_detection"] == 0).all()
assert (event_manifest.loc[animal_events, "videos_with_detection"] > 0).all()
assert event_manifest.loc[background_events, "all_videos_discarded"].all()

print(pd.crosstab(event_manifest["label_name"], event_manifest["videos_with_detection"] > 0))

videos_with_detection  False  True 
label_name                         
animal                     0    319
background                78      0


In [9]:
site_distribution = (
    event_manifest[["event_id", "label_name", "site_ids"]]
    .explode("site_ids")
    .groupby(["site_ids", "label_name"])
    .size()
    .unstack(fill_value=0)
)

site_distribution

label_name,animal,background
site_ids,,
S1,115,18
S2,74,3
S3,130,57


In [10]:
dense_summary.to_csv(DENSE_SUMMARY_PATH, index=False)
event_manifest.to_csv(EVENT_MANIFEST_PATH, index=False)
visible_intervals.to_csv(VISIBLE_INTERVALS_PATH, index=False)

print("Saved:")
print(" -", DENSE_SUMMARY_PATH)
print(" -", EVENT_MANIFEST_PATH)
print(" -", VISIBLE_INTERVALS_PATH)

Saved:
 - metadata\dense_summary.csv
 - metadata\benchmark_2_event_manifest.csv
 - metadata\benchmark_2_visible_intervals.csv


In [11]:
MAX_ANCHORS_PER_EVENT = 3
RANDOM_SEED = 42
MIN_VIDEO_DURATION_S = 2.0

positive_anchors = visible_intervals[
    ["event_id", "file_id", "start_s", "end_s"]
].copy()

positive_anchors["anchor_s"] = (
    positive_anchors["start_s"] + positive_anchors["end_s"]
) / 2
positive_anchors["anchor_source"] = "animal_detection"
positive_anchors = positive_anchors[
    ["event_id", "file_id", "anchor_s", "anchor_source"]
].drop_duplicates()

background_event_ids = set(
    event_manifest.loc[event_manifest["label_name"] == "background", "event_id"]
)
background_videos = raw_meta[raw_meta["event_id"].isin(background_event_ids)].copy()

background_rows = []
for _, row in background_videos.iterrows():
    duration = float(row["duration"])
    if duration <= 0:
        continue

    for fraction in (0.25, 0.50, 0.75):
        background_rows.append({
            "event_id": row["event_id"],
            "file_id": row["file_id"],
            "anchor_s": duration * fraction,
            "anchor_source": "background",
        })

background_anchors = pd.DataFrame(background_rows)
anchor_candidates = pd.concat(
    [positive_anchors, background_anchors],
    ignore_index=True,
)

print("Positive anchor candidates:", len(positive_anchors))
print("Background anchor candidates:", len(background_anchors))

Positive anchor candidates: 1681
Background anchor candidates: 510


In [12]:
file_metadata = raw_meta[
    ["file_id", "file_path", "site_id", "cam_id", "duration"]
].drop_duplicates("file_id").copy()

# Exact archive location used later for selective video/audio retrieval.
file_metadata["remote_video_path"] = (
    "raw_videos/" + file_metadata["file_path"].astype(str)
)

candidates = anchor_candidates.merge(
    file_metadata[["file_id", "duration"]],
    on="file_id",
    how="left",
)

primary_candidates = candidates[
    candidates["duration"] >= MIN_VIDEO_DURATION_S
].copy()

print("Events available after >= 2 s filter:", primary_candidates["event_id"].nunique())

Events available after >= 2 s filter: 389


In [13]:
sampled_groups = []

for i, (_, group) in enumerate(primary_candidates.groupby("event_id", sort=True)):
    sampled_groups.append(
        group.sample(
            n=min(MAX_ANCHORS_PER_EVENT, len(group)),
            random_state=RANDOM_SEED + i,
        )
    )

primary_anchors = pd.concat(sampled_groups, ignore_index=True)

primary_anchors = (
    primary_anchors
    .drop(columns=["duration"], errors="ignore")
    .merge(file_metadata, on="file_id", how="left")
)

event_metadata = event_manifest[
    [
        "event_id",
        "binary_label",
        "label_name",
        "species",
        "activities",
        "num_individuals",
        "split",
    ]
].drop_duplicates("event_id")

primary_anchors = primary_anchors.merge(
    event_metadata,
    on="event_id",
    how="left",
)

assert primary_anchors["duration"].notna().all()
assert primary_anchors["label_name"].notna().all()

In [14]:
def make_nested_1s_2s(anchor_s, duration_s):
    start_2s = np.clip(
        anchor_s - 1.0,
        0,
        duration_s - 2.0,
    )
    end_2s = start_2s + 2.0

    start_1s = np.clip(
        anchor_s - 0.5,
        start_2s,
        end_2s - 1.0,
    )
    end_1s = start_1s + 1.0

    return start_1s, end_1s, start_2s, end_2s


In [15]:
windows = primary_anchors.apply(
    lambda row: make_nested_1s_2s(
        float(row["anchor_s"]),
        float(row["duration"]),
    ),
    axis=1,
    result_type="expand",
)

windows.columns = [
    "start_1s",
    "end_1s",
    "start_2s",
    "end_2s",
]

primary_manifest = pd.concat(
    [
        primary_anchors.reset_index(drop=True),
        windows.reset_index(drop=True),
    ],
    axis=1,
)

In [16]:
primary_manifest = (
    primary_manifest
    .sort_values(["split", "event_id", "file_id", "anchor_s"])
    .reset_index(drop=True)
)

primary_manifest["anchor_index"] = (
    primary_manifest.groupby("event_id").cumcount()
)
primary_manifest["sample_id"] = (
    primary_manifest["event_id"].astype(str)
    + "_"
    + primary_manifest["anchor_index"].astype(str)
)

nested_ok = (
    (primary_manifest["start_2s"] <= primary_manifest["start_1s"])
    & (primary_manifest["end_1s"] <= primary_manifest["end_2s"])
)

assert nested_ok.all()
assert (
    primary_manifest.groupby("event_id")["split"].nunique().max() == 1
)

summary = (
    primary_manifest
    .groupby(["split", "label_name"])
    .agg(
        samples=("sample_id", "count"),
        events=("event_id", "nunique"),
        files=("file_id", "nunique"),
    )
)

print("All windows nested:", nested_ok.all())
print("Events retained:", primary_manifest["event_id"].nunique())
display(summary)

All windows nested: True
Events retained: 389


samples  events  files
split label_name                        
test  animal          123      57    117
      background       84      28     48
train animal          551     254    536
      background      150      50     78

In [17]:
all_events = set(event_manifest["event_id"])
primary_events = set(primary_manifest["event_id"])

excluded_events = event_manifest[
    event_manifest["event_id"].isin(all_events - primary_events)
][
    [
        "event_id",
        "split",
        "label_name",
        "species",
        "num_videos",
        "total_duration_s",
    ]
]

print("Events excluded because no source video is >= 2 s:", len(excluded_events))
excluded_events

Events excluded because no source video is >= 2 s: 8


,event_id,split,label_name,species,num_videos,total_duration_s
34,E172,train,animal,red_deer,5,5.500000
72,E231,train,animal,red_deer,5,5.500000
74,E238,train,animal,red_deer,5,6.000000
81,E251,train,animal,red_deer,13,14.500000
263,E747,train,animal,red_deer,2,2.500000
272,E759,train,animal,red_deer,4,4.500000
293,E788,train,animal,red_deer,6,5.466667
319,E138,test,animal,red_deer,7,8.500000


In [18]:
primary_manifest.to_csv(PRIMARY_MANIFEST_PATH, index=False)

print("Saved final manifest:", PRIMARY_MANIFEST_PATH)
print("Rows:", len(primary_manifest))
print("Events:", primary_manifest["event_id"].nunique())

Saved final manifest: metadata\benchmark_2_audio_primary_manifest.csv
Rows: 908
Events: 389
